In [1]:
import pandas as pd

data = pd.read_csv("reviews1.csv")

data.info()

<class 'pandas.DataFrame'>
RangeIndex: 12495 entries, 0 to 12494
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   reviewId              12495 non-null  str  
 1   userName              12495 non-null  str  
 2   userImage             12495 non-null  str  
 3   content               12495 non-null  str  
 4   score                 12495 non-null  int64
 5   thumbsUpCount         12495 non-null  int64
 6   reviewCreatedVersion  10333 non-null  str  
 7   at                    12495 non-null  str  
 8   replyContent          5818 non-null   str  
 9   repliedAt             5818 non-null   str  
 10  sortOrder             12495 non-null  str  
 11  appId                 12495 non-null  str  
dtypes: int64(2), str(10)
memory usage: 1.1 MB


In [3]:
!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     --- ------------------------------------ 1.0/12.8 MB 9.4 MB/s eta 0:00:02
     --------- ------------------------------ 3.1/12.8 MB 9.8 MB/s eta 0:00:01
     ----------------- ---------------------- 5.5/12.8 MB 10.3 MB/s eta 0:00:01
     --------------------- ------------------ 6.8/12.8 MB 10.7 MB/s eta 0:00:01
     -------------------------- ------------- 8.4/12.8 MB 9.1 MB/s eta 0:00:01
     --------------------------------- ------ 10.7/12.8 MB 9.6 MB/s eta 0:00:01
     ------------------------------------ --- 11.8/12.8 MB 8.7 MB/s eta 0:00:01
     ---------------------------------------- 12.8/12.8 MB 8.8 MB/s  0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [8]:
# Takes 1m 40s to run

import re
import spacy


# Spacy - trained pipeline package for
# en - english
# core - vocabulary, syntax, entities
# web - trained on web data (blogs, news and comments)
# sm - small size for speed, uses context sensitive tensors 
nlp = spacy.load("en_core_web_sm")


data = data[['content', 'score', 'at', 'appId']].copy()


# Dropping duplicates to avoid model memorization
data.drop_duplicates(subset=['content'], inplace = True)


# Removing any review with length < 5, to reduce noise and keep meaningful data
# Remaining 10330 rows with meaningful, removed 1477 
data = data[data['content'].str.split().str.len() >= 5].reset_index(drop=True)


def clean_text(text):
    text = text.lower()

    # Removing markdown email links 
    text = re.sub(r'\[.*?\]\(mailto:.?\)', '', text)

    # Removing plain emails
    text = re.sub(r'\S+@\S+', '', text)

    # Removing URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    # Removing HTML tags 
    text = re.sub(r'<.*?>', '', text)

    # Normalizing emojis as punctutaion (space here)
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)

    # Removing special characters (keeping apostrophes for contractions, eg: "don't")
    text = re.sub(r"[^a-z0-9\s']", ' ', text)

    # Normalizing whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text



# Using apply can make it slow for millions of rows, but okay for thousands  
data['content_clean'] = data['content'].apply(clean_text)


APP_NAME_TOKENS = {
    'anydo', 'todoist', 'task', 'habitica','forestapp' ,'forestapp', 'habitbull',
    'todos', 'timetune', 'bizcal', 'planner', 'calclock', 'habitnow', 'liferpgtasks', 'artfulagenda'
}

# Removing stopwords
DOMAIN_STOPWORDS = {
    'app', 'apps', 'please', 'thanks', 'thank', 'hi', 'hello', 'use',
    'used', 'using', 'get' , 'got', 'would', 'could', 'update', 'updated', 
    'version', 'phone', 'day', 'time', 'list', 'tap' ,'open' ,'show'
}

DOMAIN_STOPWORDS = DOMAIN_STOPWORDS | APP_NAME_TOKENS


def lemmatize(text):
    doc = nlp(text)

    # lemma_ reduces words to dictionary from; 'running','ran','runs' - 'run', 'better' - 'good'
    # .is_stop : identifies common words - this, is, at....
    # .is_alpha : identifies standard words and removes numbers, puncutations, emojis, mixed strings... 
    tokens =[
        token.lemma_ for token in doc  
        if not token.is_stop
        and token.is_alpha
        and len(token.text) > 2 # ignores very short tokens 
        and token.lemma_ not in DOMAIN_STOPWORDS 
    ] 

    return ' '.join(tokens)


data['content_processed'] = data['content_clean'].apply(lemmatize)


# Filtering length after lemmatization
data = data[data['content_processed'].str.split().str.len() >= 3].reset_index(drop=True)

print(f"Final dataset size: {len(data)}")
print(data[['content', 'content_processed']].head(10))

Final dataset size: 9495
                                             content  \
0  I have been begging for a refund from this app...   
1  Very costly for the premium version (approx In...   
2  Used to keep me organized, but all the 2020 UP...   
3  It has changed how I viewed my different lists...   
4  I'm only looking for a grocery list app but ev...   
5  Reset my free trial, new phone I'd like to see...   
6  How do to stop monthly payment because i don't...   
7  I complain about not crashes and it was immedi...   
8  Constant crashing. After reading all the negat...   
9  Widgets are useless because they always show a...   

                                   content_processed  
0                             beg refund month reply  
1  costly premium approx indian rupee year well d...  
2  organize mess thing leave enuf guess techie fe...  
3             change view different jumble find need  
4  look grocery away find way certain reopen eta ...  
5                     reset 

In [ ]:
# Checking token length distribution post-processing
lengths = data['content_processed'].str.split().str.len()
print(lengths.describe())
print(lengths.value_counts().sort_index().head(10))

RAW: I have been begging for a refund from this app for over a month and nobody is replying me
PROCESSED: beg refund month reply
---
RAW: Very costly for the premium version (approx Indian Rupees 910 per year). Better to download the premium version of this app from apkmos website and use it. Microsoft to do list app is far more better.
PROCESSED: costly premium approx indian rupee year well download premium apkmos website microsoft list far well
---
RAW: Used to keep me organized, but all the 2020 UPDATES have made a mess of things !!! Y cudn't u leave well enuf alone ??? Guess ur techies feel the need to keep making changes to justify continuing to collect their salary !!! 🤤🤤🤤
PROCESSED: organize mess thing leave enuf guess techie feel need make change justify continue collect salary
---
RAW: It has changed how I viewed my different lists. Now they are all jumbled together and I can't find what I need.
PROCESSED: change view different list jumble find need
---
RAW: I'm only looking f